# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring.**

I picked this lane because the decision is concrete — which page an editor opens first — and the starter dataset is this lane's default dataset, so I can start now without waiting on warehouse access. It also ships a baseline I can measure myself: the hand rule in `scripts/02_baseline_score.py` scores 0.240 Precision@50, worse than random ordering (Section 3). That gives me a clear gap to work on.

Provisional until the end of Week 4, as the card allows.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**My question:** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis.** One page — a single content item, which is one row in the starter dataset, measured over a trailing 90-day window. Not a client and not a day: I score pages, and a client is only the grouping I split on for validation so no client's pages land in both train and test.

**The output.** A ranked review queue: one row per page, with a priority score, a suggested action, a reason code, and a confidence label.

**The decision it improves.** Not "predict decline" — the *order* of that queue. An editor has limited hours and a backlog much bigger than those hours (Section 3), so they will review some pages either way. My work only changes which ones come first.

**Who acts, and how.** A content/SEO editor working across client sites. They take the top N pages their week allows and pick one action per page: refresh, expand, protect, prune, or monitor. Each row needs a reason code, since nobody should act on a bare score.

**What a wrong call costs.** A false positive wastes an editor's time on a healthy page — a few hours, and recoverable. A false negative leaves a declining page with real traffic to keep sliding for another quarter, and nobody finds out it was missed. The false negative is worse, so I will not judge my work on Precision@50 alone.

**Why not just an if-statement.** The repo already contains one, and I measured it: 0.240 Precision@50 against a 0.391 base rate — worse than random. Per-client decline rates also run from 0.00 to 0.94 (Section 3), so no single global threshold fits all 32 clients. The pattern is real but too tangled to hand-write, which is where ML earns its place. If the hand rule had scored 0.60, the honest answer would have been to tune the rule instead.

**Task type:** ranking/scoring. **Metric, named now:** Precision@50, validated with a client holdout. **Never features:** `trend_direction` and `trend_pct`, since the label is derived from them.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import json, os
from pathlib import Path
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declining"] = df["trend_direction"].str.lower().eq("down")
print(f"{len(df):,} pages | {df['client_id'].nunique()} pseudonymized clients\n")

# 1. The backlog is far bigger than anyone can review -> the ORDER is the problem.
backlog = df[df["declining"] & (df["impressions_90d"] >= 500)]
print(f"1. Declining AND impressions_90d >= 500: {len(backlog):,} pages")
print(f"   = {len(backlog) / 50:,.0f} weeks of queue at 50 reviews/week\n")

# 2. The shipped hand rule is worse than random at K=50. (Needs run_all.py first.)
if Path("outputs/model_results.json").exists():
    r = json.loads(Path("outputs/model_results.json").read_text())
    preds = pd.read_csv("data/processed/model_predictions.csv")
    holdout_rate = preds.loc[preds["split"] == "test", "is_declining_label"].mean()
    print(f"2. Hand-rule baseline Precision@50: {r['baseline']['baseline_precision_at_50']:.3f}")
    print(f"   Holdout base rate (random order): {holdout_rate:.3f}  <- the bar to clear\n")
else:
    print("2. Run `python scripts/run_all.py` first (outputs/ is gitignored).\n")

# 3. "Declining" is not on one global scale across clients.
per_client = df.groupby("client_id")["declining"].agg(["size", "mean"])
per_client = per_client[per_client["size"] >= 50]
print(f"3. Per-client decline rate ({len(per_client)} clients with >= 50 pages):")
print(f"   min {per_client['mean'].min():.3f} | median {per_client['mean'].median():.3f} "
      f"| max {per_client['mean'].max():.3f}")


30,000 pages | 32 pseudonymized clients

1. Declining AND impressions_90d >= 500: 9,961 pages
   = 199 weeks of queue at 50 reviews/week

2. Hand-rule baseline Precision@50: 0.240
   Holdout base rate (random order): 0.391  <- the bar to clear

3. Per-client decline rate (25 clients with >= 50 pages):
   min 0.000 | median 0.549 | max 0.937


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say.**

- *Observed:* "In a 90-day anonymized snapshot of 30,000 pages across 32 pseudonymized clients, 9,961 pages were declining and still earning 500+ impressions."
- *Directional:* "The learned ranking placed more declining pages in its top 50 than the hand rule did, across repeated client-holdout splits."
- *Decision-support:* "This queue suggests which pages an editor might open first, with a reason code on each row."

**What I can never say.**

- No causal claims. I have no intervention data, so "refreshing these pages will recover traffic" is off limits — I can only observe that declining pages share certain properties.
- Nothing about Google's algorithm. I am describing patterns in one agency's measured data, not reverse-engineering a ranking system.
- My label is `trend_direction`, a bucket already computed in the shipped data, so I am predicting a *recorded trend bucket*, not future traffic. I will state that limitation rather than let it be assumed.
- No false precision. Numbers get quoted with their spread across splits, not to three decimals.
- Nothing identifying: `content_id` and `client_id` are pseudonyms used for grouping only, and no client names, URLs, or queries appear anywhere.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
